# Experiment 11 — first CARES-IDS specialist-method run

This is the **first actual proposed-method training run**, not another baseline. It reuses the frozen weighted-LightGBM baseline (63.669% ten-class macro-F1 on original 35,005 internal validation targets), trains **three small specialist classifiers** on the designated original **training split only**, and tests a fixed probability-mass-preserving selective classifier.

**Three clinically relevant confusion pairs:** DoS/Exploits, Backdoor/Exploits, Analysis/Exploits. Each specialist has disjoint 70% pair-fitting, 15% early stopping and 15% probability calibration partitions from the training fold. Their outputs can redistribute only the baseline probability allocated to their own class pair. Gate hyperparameters were fixed before training; there are **six comparisons**, including all-gated vs no-gate and single-pair ablations. Outputs include private 10-class probabilities, raw 10×10 confusion CSVs, per-class F1, and the learned binary specialist checkpoints.

**Evidence boundary:** This is development validation, not official test. Confusion pairs were chosen based on development-validation findings, so results must not be represented as unbiased. Conditional-specialist/gated-routing ideas have prior work; novelty is provisional until differentiated from 2024–2026 literature. The code never opens the official test set.

**Requirements:** Existing Kaggle `/kaggle/working/research_outputs/CTRL-TAB-001-S42` run and frozen `SCV-IDS-REPRO-001` split must still be present. Keep the same `dsshayan/unsw-nb15` dataset. GPU not needed. No need for `pushRare` secret until archiving in the following step.

In [ ]:
# ONE CELL: fit actual confusion specialists against frozen, hash-checked LightGBM.
from pathlib import Path
import subprocess, sys, os
REPO=Path("/kaggle/working/rare-attack-ids-fast-tabular")
BASE=Path("/kaggle/working/research_outputs")
OLD=BASE/"CTRL-TAB-001-S42"
NEW=BASE/"CARES-PAIR-001-S42"
REF=BASE/"SCV-IDS-REPRO-001"
for p in [OLD/"lgbm_rare_sqrt"/"model.txt",
          OLD/"lgbm_rare_sqrt"/"predictions_PRIVATE.npz",
          REF/"split_indices.npz",
          REF/"validation_predictions.npz"]:
    assert p.is_file(), f"Source experiment not in current Kaggle session: {p}. Stop; restore archives first."

subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
print("Research code:",subprocess.check_output(
    ["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip())

env=os.environ.copy()
env["PYTHONPATH"]=str(REPO/"src")
env["OMP_NUM_THREADS"]="4"
env["OPENBLAS_NUM_THREADS"]="1"
env["MPLBACKEND"]="Agg"

subprocess.run([
    sys.executable,str(REPO/"scripts/run_cares_pair_prototype.py"),
    "--repo",str(REPO),
    "--config",str(REPO/"configs/cares_pair_prototype_seed42.json"),
    "--reference",str(REF),
    "--baseline",str(OLD),
    "--output",str(NEW)
],cwd=str(REPO),env=env,check=True)

print("\nACTUAL PROPOSED-METHOD RESULTS")
print((NEW/"comparison_scores.csv").read_text())

**Next:** Paste the `CARES FIRST PROPOSED-METHOD EXPERIMENT COMPLETE` section. Do not tune the official test or repeat training. Save the Kaggle session; we'll independently verify all six measured confusion matrices and archive the trained specialists in a new unpublished draft GitHub Release before designing a stronger follow-up.